# Tree of Thoughts: Phase 1 - Basic Concepts

This notebook provides a comprehensive exploration of Tree of Thoughts (ToT) methodology, including conceptual foundations, prompt engineering strategies, and practical implementation patterns.

**Table of Contents:**
1. Introduction to Tree of Thoughts
2. Agent vs Model: When Do We Use ToT?
3. Prompt Engineering for Thought Generation
4. Prompt Engineering for Thought Evaluation
5. Side-by-Side Prompt Comparisons
6. Practical Code Examples
7. Complete Working Example

## 1. Introduction to Tree of Thoughts

Tree of Thoughts (ToT) is a framework for solving complex problems by:
1. **Decomposing problems** into intermediate steps (thoughts)
2. **Generating multiple candidates** at each step
3. **Evaluating candidates** to guide exploration
4. **Building a tree structure** of the solution space

### Key Insights

**Why Trees Instead of Chains?**
- Chain of Thought (CoT) explores one linear path: Problem → Step1 → Step2 → Step3 → Answer
- Tree of Thoughts explores multiple paths: Problem → [Step1a, Step1b, Step1c] → [Step2a, Step2b, ...] → Answer
- Trees allow **backtracking** and **exploration** of the solution space
- Trees help avoid **dead ends** by evaluating multiple branches

**When Trees Help Most:**
- Problems with **multiple solution paths** (creative writing, math, coding)
- Problems requiring **exploration and planning** (games, logistics, design)
- Problems where **evaluation is easier than generation** (code review, story critique)
- Problems with **high complexity** and **interdependent choices**

## 2. Agent vs Model: When Do We Use ToT?

There are two fundamentally different ways to implement Tree of Thoughts:

### 2.1 In-Context ToT (Model-Based)

**How It Works:**
- You give a **single prompt** to the language model
- The prompt tells the model to think like a tree explorer
- The model **internally simulates** the tree structure
- The model returns **one final answer**

**Example Prompt:**
```
Use tree of thoughts to solve this problem:
[problem]

Generate 3 candidate approaches. Evaluate each. Then generate 3 approaches
for the most promising one. Continue until you have a solution.
```

**Characteristics:**
| Aspect | Details |
|--------|----------|
| **API Calls** | 1 call per problem |
| **Cost** | Low (single API call) |
| **Control** | Model decides exploration strategy |
| **Speed** | Fast (one round-trip) |
| **Quality** | Depends on prompt clarity and model capability |
| **Debugging** | Hard to see intermediate reasoning |
| **Best For** | Simple problems, quick iteration, cost-sensitive applications |

**Pros:**
- ✓ Simple to implement
- ✓ Single API call
- ✓ Lower cost
- ✓ Fast execution
- ✓ No orchestration needed

**Cons:**
- ✗ Model may not explore tree deeply
- ✗ Can't control which branches to expand
- ✗ Hard to see what the model actually considered
- ✗ Limited ability to restart from intermediate states
- ✗ May just do CoT in the prompt

---

### 2.2 Agentic ToT (System-Based)

**How It Works:**
- You write an **agent/program** that orchestrates multiple LLM calls
- Each **node** in the tree = separate LLM call
- The **agent** controls:
  - Which branches to expand
  - When to stop exploring
  - How to prune unpromising paths
- The **agent** returns the best path found

**Example Flow:**
```python
# Agent orchestration
tree_root = Node(problem)
queue = [tree_root]

while queue and budget_remaining:
    current_node = queue.pop()
    
    # LLM Call 1: Generate candidates
    candidates = llm.generate_candidates(current_node)
    
    for candidate in candidates:
        child_node = Node(candidate, parent=current_node)
        
        # LLM Call 2: Evaluate candidate
        score = llm.evaluate_candidate(child_node)
        
        if score > threshold:
            queue.append(child_node)

return best_path_found
```

**Characteristics:**
| Aspect | Details |
|--------|----------|
| **API Calls** | Multiple calls (can be 10s-100s) |
| **Cost** | Higher (many API calls) |
| **Control** | Agent controls exploration strategy |
| **Speed** | Slower (multiple round-trips) |
| **Quality** | Higher (systematic exploration) |
| **Debugging** | Easy to see every node and decision |
| **Best For** | Complex problems, research, high-stakes decisions |

**Pros:**
- ✓ Full control over exploration
- ✓ Can see every decision point
- ✓ Easy to implement search algorithms (BFS, DFS, beam search)
- ✓ Can restart from any node
- ✓ Better quality through systematic exploration
- ✓ Can set budgets and constraints

**Cons:**
- ✗ Many API calls (expensive)
- ✗ Slower execution
- ✗ More code to write and maintain
- ✗ Requires orchestration logic
- ✗ Risk of prompt inconsistency across calls

---

### 2.3 When to Use Each Approach

**Use In-Context ToT (Model-Based) When:**
- Problem is relatively simple
- You need fast iteration during development
- Budget is tight
- The model is very capable (GPT-4, Claude)
- You're doing one-off problem solving

**Use Agentic ToT (System-Based) When:**
- Problem is complex with many interdependencies
- You need reproducible, deterministic behavior
- You want to analyze the solution process
- Quality is more important than cost
- You need to integrate with other systems
- You're building a production application
- You want to use beam search or other algorithms

---

### 2.4 Trade-offs and Comparisons

```
                        In-Context ToT    vs    Agentic ToT
                        
Complexity              Simple                   Complex
├─ Code                 10-20 lines              100-500 lines
├─ Setup                Minutes                  Hours
└─ Debugging            Hard                     Easy

Cost                    Cheap                    Expensive
├─ Per problem          ~$0.01-0.10              ~$0.10-1.00
└─ At scale             Linear with volume       Multiplicative with depth

Quality                 Medium                   High
├─ Exploration          Uncontrolled             Controlled
├─ Depth                Limited                  Tuneable
└─ Reproducibility      Variable                 Consistent

Speed                   Fast                     Slow
├─ Response time        Seconds                  Minutes
└─ Iterations           Quick                    Slow feedback loop

Control                 None                     Full
├─ Branch selection     Model decides            Agent decides
├─ Depth control        Implicit                 Explicit
└─ Restart from node    Not possible             Easy
```

## 3. Prompt Engineering for Thought Generation

The quality of candidates directly affects the quality of solutions. Good prompts generate diverse, relevant, and evaluable candidates.

### 3.1 Bad Prompts (Generic, Vague)

❌ **Example 1: Too Generic**
```
Generate some ideas for solving this problem:
[problem]
```

Problems:
- No guidance on diversity or structure
- No examples of good candidates
- Model may generate irrelevant or trivial ideas
- Hard to evaluate candidates consistently

❌ **Example 2: Too Vague**
```
Think about how to solve:
[problem]
```

Problems:
- "Think about" is not a clear instruction
- Model may just explain, not generate alternatives
- No format specification
- Inconsistent output structure

❌ **Example 3: Biased Toward One Approach**
```
Think of a clever mathematical approach to:
[problem]
```

Problems:
- Primes the model toward math solutions
- Might miss non-mathematical approaches
- Reduces diversity of candidates

---

### 3.2 Good Prompts (Specific, Structured)

✓ **Strong Structure Elements:**

1. **Clear role/context**: "You are a problem-solving expert"
2. **Specific instruction**: "Generate N distinct approaches"
3. **Diversity guidance**: "Approaches should be different from each other"
4. **Format specification**: "Return as a numbered list"
5. **Quality criteria**: "Each approach should be concrete and feasible"
6. **Example (optional)**: Show what good output looks like

✓ **Example: Strong Prompt**
```
You are an expert problem solver specializing in [domain].

For this problem:
[problem]

Generate 3 DISTINCT approaches. Each approach should:
- Be different from the others (avoid duplicates)
- Be concrete and actionable
- Be feasible to evaluate or execute
- Include a brief rationale

Format each approach as:
Approach 1: [approach name]
Rationale: [why this might work]
Next Steps: [2-3 specific steps]

Be creative and consider unconventional approaches, not just the obvious ones.
```

---

### 3.3 Domain-Specific Variations

#### 3.3a Math Problems (Game of 24, Equations)

```
You are an expert mathematician.

Problem: Make 24 from [numbers]

Generate 3 distinct calculation approaches:
- Each approach should use a different grouping or operation sequence
- Each should be concrete (show intermediate results)
- Each should indicate if it reaches 24 or not

Format:
Approach 1:
Step 1: [calculation] = [result]
Step 2: [calculation] = [result]
...
Final Result: [number]

Prioritize:
1. Different operation orderings (not just different numbers)
2. Approaches that might work (not obviously wrong)
3. Clear intermediate steps
```

#### 3.3b Creative Writing (Story Generation)

```
You are an accomplished fiction writer.

Story Setup: [premise]

Generate 3 distinct narrative directions for the next scene:
- Each should take the story in a different thematic direction
- Each should introduce new elements or reveal hidden information
- Each should be engaging and move the plot forward

Format each as:
Direction 1: [short title]
Narrative Arc: [2-3 sentence description of how the scene unfolds]
Key Elements: [new characters, reveals, or conflicts introduced]
Emotional Tone: [primary emotion conveyed]

Ensure each direction:
- Is different from the others
- Builds on established story elements
- Offers interesting possibilities for continuation
```

#### 3.3c Code Debugging

```
You are an expert code reviewer and debugger.

Problematic Code:
[code]

Expected Behavior: [what should happen]
Actual Behavior: [what's happening instead]

Generate 3 distinct hypotheses about the root cause:
- Each hypothesis should point to a different category of bug
- Each should be specific and testable
- Each should explain the observed behavior

Format:
Hypothesis 1: [bug category and specific issue]
Why This Explains the Behavior: [2-3 sentences]
Code Location: [which lines are involved]
How to Test: [specific test or inspection]

Categories to consider:
- Logic errors (incorrect condition or flow)
- State management (variable initialization or scope)
- Type/data errors (wrong type or format)
- Performance issues (infinite loops, memory leaks)
- External dependencies (API calls, file access)
```

#### 3.3d Logical Reasoning

```
You are a master of logical reasoning and deduction.

Problem: [logical puzzle or reasoning task]

Generate 3 distinct reasoning approaches:
- Each should start from different premises or evidence
- Each should follow a clear logical chain
- Each should lead to a specific conclusion

Format:
Approach 1: [approach name]
Starting Point: [initial premise or observation]
Logical Chain: 
  1. [deduction] → [conclusion]
  2. [deduction] → [conclusion]
  ...
Final Conclusion: [what this approach concludes]
Confidence: [high/medium/low and why]

Ensure approaches differ in:
- Starting assumptions
- Evidence prioritized
- Logical pathway
```

## 4. Prompt Engineering for Thought Evaluation

Evaluating candidates without solving the entire problem is critical to ToT efficiency.

### 4.1 Evaluation Without Full Solutions

**Key Insight:** You don't need to complete a solution to assess if an intermediate thought is promising.

**Evaluation Criteria:**
1. **Feasibility**: Can this step be completed with reasonable effort?
2. **Progress**: Does this step move toward the goal?
3. **Clarity**: Is the next step obvious?
4. **Consistency**: Does this fit with previous steps?
5. **Novelty**: Does this explore a different path?

**What NOT to Evaluate:**
- ❌ "Will this definitely work?" (You don't know yet)
- ❌ "Is this the optimal solution?" (You're not there yet)
- ❌ "Is this perfect?" (It's intermediate, not final)

---

### 4.2 Prompts for Different Evaluation Criteria

#### 4.2a Feasibility Evaluation

```
You are an expert evaluator of problem-solving approaches.

Context: [problem statement]
Proposed Step: [the candidate thought]

Evaluate the FEASIBILITY of this step:
- Can it be executed given current constraints?
- Are required resources available?
- Is it technically possible?
- Will it take reasonable time/effort?

Score 0-1: [score]
Reasoning: [2-3 sentences]
```

#### 4.2b Progress Evaluation

```
You are an expert evaluator of problem-solving progress.

Original Problem: [problem]
Goal: [desired outcome]
Progress So Far: [what's been done]
Proposed Next Step: [the candidate thought]

Evaluate if this step makes MEANINGFUL PROGRESS:
- Does it move closer to the goal?
- Does it resolve key unknowns?
- Does it eliminate dead ends?
- Does it build on previous progress?

Score 0-1: [score]
Reasoning: [2-3 sentences]
```

#### 4.2c Clarity Evaluation

```
You are an expert evaluator of clear reasoning.

Problem: [problem]
Current Thought: [the candidate thought]

Evaluate the CLARITY of this thought:
- Is it well-defined and unambiguous?
- Are next steps obvious from here?
- Could someone continue from this point?
- Does it avoid vagueness?

Score 0-1: [score]
Reasoning: [2-3 sentences]
If unclear, how could it be improved?
```

#### 4.2d Composite Evaluation

```
You are an expert evaluator of intermediate solution steps.

Problem: [problem]
Previous Steps: [what's been done]
Candidate Thought: [the candidate thought]

Evaluate this thought on multiple criteria:

1. FEASIBILITY (0-1):
   - Can it be executed?
   - Score: [score]
   - Reason: [brief]

2. PROGRESS (0-1):
   - Does it move toward the goal?
   - Score: [score]
   - Reason: [brief]

3. CLARITY (0-1):
   - Is it well-defined?
   - Score: [score]
   - Reason: [brief]

4. CONSISTENCY (0-1):
   - Does it fit with previous steps?
   - Score: [score]
   - Reason: [brief]

OVERALL SCORE (average of above): [score]
RECOMMENDATION: [pursue / explore further / deprioritize]
```

---

### 4.3 Rubrics and Scoring Guidelines

**5-Point Rubric (for simplicity):**
```
5 (Excellent)
  - Clearly makes significant progress
  - Well-defined and clear
  - Feasible to execute
  - Opens up multiple continuations

4 (Good)
  - Makes clear progress
  - Mostly clear with minor ambiguity
  - Feasible
  - Provides direction for next steps

3 (Acceptable)
  - Makes some progress
  - Somewhat clear but needs refinement
  - Feasible but may require effort
  - Next steps not obvious

2 (Questionable)
  - Minimal progress or progress unclear
  - Unclear or vague
  - Feasibility questionable
  - Difficult to continue from here

1 (Poor)
  - No clear progress toward goal
  - Unclear or incoherent
  - Likely infeasible
  - Cannot continue from here
```

**0-1 Scale (for continuous scoring):**
```
0.0-0.2:  Unlikely to be useful; consider pruning
0.2-0.4:  Possible but risky; low priority
0.4-0.6:  Acceptable but not strong; medium priority
0.6-0.8:  Good; worthy of exploration
0.8-1.0:  Excellent; high priority for expansion
```

---

### 4.4 Examples: Showing Scoring

#### Example 1: Math Problem Scoring

```
Problem: Make 24 from [3, 8, 3, 8]

Candidate 1: Start with 3 * 8 = 24, then use remaining numbers
Score: 0.9
Reasoning: Immediately finds a product equal to the goal. Just need to
verify the remaining numbers (3, 8) don't interfere. High feasibility.

Candidate 2: Try to get close to 24 with multiplication
Score: 0.4
Reasoning: Vague approach. "Close to 24" isn't precise. Unclear what
operations to try. Could work but low progress clarity.

Candidate 3: Add all numbers and see if we can subtract to 24
Score: 0.2
Reasoning: 3+8+3+8=22, so subtracting won't reach 24. Dead end.
```

#### Example 2: Code Debugging Scoring

```
Problem: Function returns None instead of list

Hypothesis 1: The function returns early before building the list
Score: 0.85
Reasoning: Very specific and testable. Directly explains the symptom.
Easy to check by adding debug logs. High feasibility of finding the issue.

Hypothesis 2: There's a logic error
Score: 0.3
Reasoning: Too vague. "Logic error" could mean anything. Doesn't guide
the search effectively. Low progress if this is all you know.

Hypothesis 3: The variable is shadowed somewhere
Score: 0.6
Reasoning: Plausible and specific enough to test. Variable shadowing
could cause this. Testable but requires checking scope carefully.
```

## 5. Side-by-Side Prompt Comparisons

### 5.1 Chain of Thought vs Tree of Thoughts

Let's solve the same problem with both approaches to see the difference.

**Problem:** "I need to plan a company outing. 50 people, $2000 budget, indoor due to weather, needs to be team-building."

---

### 5.2 Chain of Thought Approach (Linear)

**Prompt:**
```
You are an event planner. Plan a company outing.

Requirements:
- 50 people
- $2000 total budget ($40/person)
- Indoor (weather constraint)
- Team-building focus

Think through this step by step:
1. What venue type would work?
2. What activities would build team?
3. What's the timeline?
4. What's the cost breakdown?

Provide a complete plan.
```

**Expected Output (Linear Path):**
```
Step 1: Venue
→ Choose a bowling alley (affordable, team-building)

Step 2: Activities
→ Bowling tournament (people already there)
→ Pizza lunch (included in budget)

Step 3: Timeline
→ 2:00pm - 5:00pm Saturday

Step 4: Budget
→ Bowling: $1000 (20 lanes × $50)
→ Pizza: $800 (40 pizzas)
→ Prizes: $200
→ Total: $2000

PLAN: Saturday bowling outing with pizza and prizes.
```

**Characteristics:**
- Follows one predetermined sequence
- Once venue chosen (bowling), all downstream decisions follow from it
- If bowling is suboptimal, the entire plan is affected
- Model doesn't explore other venues

---

### 5.3 Tree of Thoughts Approach (Branching)

**Prompt:**
```
You are an expert event planner.

Task: Plan a company outing with:
- 50 people
- $2000 budget ($40/person)
- Indoor venue (weather constraint)
- Team-building activities

Generate 3 DISTINCT venue types, each starting a different path:

For each venue, include:
1. Venue Type: [type and examples]
2. Why This Works: [why it's suitable]
3. Core Activities: [what fits this venue]
4. Budget Breakdown: [rough estimate]
5. Feasibility Score (0-1): [likelihood of success]

Venues to explore:
- Option A: Entertainment venue (bowling, arcade, etc)
- Option B: Workshop/Class venue (cooking class, escape room, etc)
- Option C: Multi-space venue (restaurant with private room, conference center)

Be specific in details for each path.
```

**Expected Output (Multiple Branches):**
```
BRANCH 1: Entertainment Venue (Bowling Alley)
├─ Venue: Local bowling alley with arcade
├─ Activities: Tournament → Pizza dinner → Arcade challenge
├─ Budget: Lanes $1000, Food $800, Prizes $200
└─ Feasibility: 0.85 (standard, proven approach)

BRANCH 2: Workshop Venue (Escape Room + Dinner)
├─ Venue: Multiple escape rooms + restaurant
├─ Activities: 6 teams rotate escape rooms → Group dinner
├─ Budget: Rooms $1200, Dinner $800
└─ Feasibility: 0.75 (high engagement but complex logistics)

BRANCH 3: Multi-Space Venue (Conference Center)
├─ Venue: Hotel conference center with breakout rooms
├─ Activities: Morning workshops → Lunch → Team challenges → awards
├─ Budget: Space $300, Catering $1400, Materials $300
└─ Feasibility: 0.70 (most flexible but requires more planning)

Each branch leads to different downstream decisions.
Branch 2 requires transportation planning.
Branch 3 requires multiple facilitators.
```

**Characteristics:**
- Explores multiple venue types in parallel
- Each venue opens different possibilities downstream
- Easier to compare trade-offs:
  - Branch 1: Simplest, lowest risk
  - Branch 2: Most engaging, complex
  - Branch 3: Most flexible, requires more work
- Decision-maker can see all options before committing

---

### 5.4 Exploring Further Down the Tree

**Second Level Prompt (expand most promising branch):**
```
You evaluated three venue options. The escape room option (Branch 2)
has highest engagement but raised complexity concerns.

Deep dive into: Escape Room + Dinner approach

Generate 3 ways to handle the complexity:

A) Simplify Logistics
   - Single escape room (multiple seatings)
   - Nearby restaurant
   - Minimizes transportation

B) Embrace Multiple Locations
   - 6 different rooms (teams rotate)
   - Hire professional coordinators
   - Higher engagement

C) Hybrid Approach
   - 2-3 rooms in same complex
   - Pre-assign rooms to balance groups
   - Integrated dinner in complex

For each, provide logistics details and revised feasibility.
```

**Result:**
```
This two-level tree comparison shows:
- Level 1: Venue type (3 branches)
- Level 2: Logistical approach (3 variations for chosen branch)

The tree allows the decision-maker to:
1. Choose best venue type
2. Solve logistical concerns within that choice
3. See trade-offs at each decision point
```

---

### 5.5 Key Differences Summarized

| Aspect | Chain of Thought | Tree of Thoughts |
|--------|-----------------|------------------|
| **Paths Explored** | 1 linear path | Multiple branches |
| **Decision Visibility** | Sequence of decisions | Decision tree with alternatives |
| **Backtracking** | Not possible | Can explore alternatives |
| **Trade-off Analysis** | Shows one plan | Shows multiple plans side-by-side |
| **Quality** | Depends on first choice | Robust across options |
| **Decision-Making** | "Here's the answer" | "Here are your options" |
| **Complexity** | Simple | Harder to follow |
| **Best For** | Simple linear tasks | Complex decisions with trade-offs |

## 6. Practical Code Examples

This section provides reusable classes for managing prompts in a structured way.

### 6.1 PromptTemplate for Generation

In [ ]:
from typing import List, Dict
from dataclasses import dataclass
import json

@dataclass
class GenerationPromptTemplate:
    """
    Template for generating candidate thoughts.
    
    Attributes:
        name: Identifier for this prompt template
        role: Expert role to assume
        problem_description: How to describe the problem
        num_candidates: How many candidates to generate
        diversity_instruction: How to ensure diversity
        format_instruction: How to format output
        quality_criteria: What makes a good candidate
    """
    name: str
    role: str
    problem_description: str
    num_candidates: int
    diversity_instruction: str
    format_instruction: str
    quality_criteria: List[str]
    
    def build_prompt(self, problem: str, context: Dict[str, str] = None) -> str:
        """
        Build the complete prompt from template.
        
        Args:
            problem: The specific problem to solve
            context: Additional context variables (domain, constraints, etc)
        
        Returns:
            Complete prompt string ready for LLM
        """
        prompt = f"""You are a {self.role}.

{self.problem_description}:
{problem}

Generate {self.num_candidates} DISTINCT candidates for solving this problem.

Diversity Requirement:
{self.diversity_instruction}

Quality Criteria:
Each candidate should:
"""
        
        for criterion in self.quality_criteria:
            prompt += f"\n- {criterion}"
        
        prompt += f"\n\n{self.format_instruction}"
        
        # Add optional context
        if context:
            prompt += "\nAdditional Context:\n"
            for key, value in context.items():
                prompt += f"- {key}: {value}\n"
        
        return prompt
    
    def __repr__(self) -> str:
        return f"GenerationPrompt({self.name}, {self.num_candidates} candidates)"


# Example: Math Problem Template
math_template = GenerationPromptTemplate(
    name="game_of_24",
    role="expert mathematician specializing in numerical puzzles",
    problem_description="Problem: Make 24 from the given numbers",
    num_candidates=3,
    diversity_instruction="Each approach should use different groupings or operation sequences.",
    format_instruction="""Format each candidate as:
Candidate N: [approach name]
Step 1: [calculation] = [intermediate result]
Step 2: [calculation] = [intermediate result]
...
Final Result: [number]
Viability: [Does this reach 24? Yes/No]""",
    quality_criteria=[
        "Use different operation orderings (not just reordering numbers)",
        "Show all intermediate steps",
        "Clearly indicate if it reaches 24 or not",
        "Be concrete and specific"
    ]
)

# Build a prompt for specific numbers
problem_numbers = "[3, 8, 3, 8]"
context = {"constraint": "Must use each number exactly once",
             "operations": "Can use +, -, *, /, and parentheses"}

prompt = math_template.build_prompt(problem_numbers, context)
print("Generated Prompt for Math Problem:")
print("=" * 60)
print(prompt)
print("=" * 60)

### 6.2 PromptTemplate for Evaluation

In [ ]:
@dataclass
class EvaluationPromptTemplate:
    """
    Template for evaluating candidate thoughts.
    
    Attributes:
        name: Identifier for this prompt template
        role: Expert evaluator role
        criteria: List of evaluation criteria
        scoring_scale: How to score (e.g., "0-1" or "1-5")
        scoring_guidelines: Explanation of scores
    """
    name: str
    role: str
    criteria: List[str]
    scoring_scale: str
    scoring_guidelines: Dict[str, str]
    
    def build_prompt(self, problem: str, candidate: str, context: Dict[str, str] = None) -> str:
        """
        Build evaluation prompt.
        
        Args:
            problem: Original problem statement
            candidate: The candidate thought to evaluate
            context: Additional context (previous steps, goals, etc)
        
        Returns:
            Complete evaluation prompt
        """
        prompt = f"""You are a {self.role}.

Your task: Evaluate the following candidate thought.

Original Problem:
{problem}

Candidate Thought to Evaluate:
{candidate}
"""
        
        if context:
            prompt += "\nContext:"
            for key, value in context.items():
                prompt += f"\n- {key}: {value}"
        
        prompt += f"\n\nEvaluate on the following criteria:\n"
        
        for criterion in self.criteria:
            prompt += f"\n{criterion}:"
            prompt += f"\n  Score ({self.scoring_scale}): [score]"
            prompt += f"\n  Reasoning: [brief explanation]\n"
        
        prompt += f"\n\nScoring Guidelines ({self.scoring_scale}):\n"
        for score_level, explanation in self.scoring_guidelines.items():
            prompt += f"\n{score_level}: {explanation}"
        
        prompt += "\n\nProvide your evaluation above, then give OVERALL SCORE and RECOMMENDATION."
        
        return prompt
    
    def __repr__(self) -> str:
        return f"EvaluationPrompt({self.name}, {len(self.criteria)} criteria)"


# Example: Math Evaluation Template
math_eval_template = EvaluationPromptTemplate(
    name="math_solution_viability",
    role="expert mathematician evaluating solution approaches",
    criteria=[
        "Correctness of Arithmetic",
        "Clear Logical Steps",
        "Proper Use of Operations",
        "Path to Goal (does it reach 24?)"
    ],
    scoring_scale="0-1",
    scoring_guidelines={
        "0.0-0.2": "Approach is incorrect or unfeasible",
        "0.2-0.4": "Approach has significant issues but some merit",
        "0.4-0.6": "Approach is acceptable but has concerns",
        "0.6-0.8": "Approach is good with minor issues",
        "0.8-1.0": "Approach is excellent and clearly viable"
    }
)

# Build evaluation prompt for a specific candidate
problem = "Make 24 from [3, 8, 3, 8]"
candidate = """Step 1: 8 / (3 - 8/3) = 8 / (1/3) = 24
Final Result: 24"""

eval_context = {
    "Goal": "Reach exactly 24",
    "Constraint": "Use each number exactly once"
}

eval_prompt = math_eval_template.build_prompt(problem, candidate, eval_context)
print("Generated Evaluation Prompt:")
print("=" * 60)
print(eval_prompt)
print("=" * 60)

### 6.3 Managing Multiple Prompts per Domain

In [ ]:
class PromptRegistry:
    """
    Registry to manage multiple prompt templates by domain.
    """
    
    def __init__(self):
        self.generation_prompts = {}
        self.evaluation_prompts = {}
    
    def register_generation_prompt(self, template: GenerationPromptTemplate):
        """Register a generation prompt template."""
        self.generation_prompts[template.name] = template
    
    def register_evaluation_prompt(self, template: EvaluationPromptTemplate):
        """Register an evaluation prompt template."""
        self.evaluation_prompts[template.name] = template
    
    def get_generation_prompt(self, name: str) -> GenerationPromptTemplate:
        """Get a registered generation prompt."""
        if name not in self.generation_prompts:
            raise ValueError(f"Generation prompt '{name}' not found")
        return self.generation_prompts[name]
    
    def get_evaluation_prompt(self, name: str) -> EvaluationPromptTemplate:
        """Get a registered evaluation prompt."""
        if name not in self.evaluation_prompts:
            raise ValueError(f"Evaluation prompt '{name}' not found")
        return self.evaluation_prompts[name]
    
    def list_generation_prompts(self) -> List[str]:
        """List all available generation prompts."""
        return list(self.generation_prompts.keys())
    
    def list_evaluation_prompts(self) -> List[str]:
        """List all available evaluation prompts."""
        return list(self.evaluation_prompts.keys())


# Create a registry and populate with examples
registry = PromptRegistry()

# Register math prompts
registry.register_generation_prompt(math_template)
registry.register_evaluation_prompt(math_eval_template)

# Example: Register a coding template
coding_gen_template = GenerationPromptTemplate(
    name="code_debugging",
    role="expert code reviewer and debugger",
    problem_description="Debug this code",
    num_candidates=3,
    diversity_instruction="Each hypothesis should point to a different category of bug.",
    format_instruction="""For each hypothesis:
- Name: [bug category]
- Code Location: [which lines]
- Why This Explains It: [explanation]
- How to Test: [test method]""",
    quality_criteria=[
        "Point to different bug categories",
        "Be specific about code locations",
        "Explain how it causes the observed behavior",
        "Be testable with concrete verification steps"
    ]
)

registry.register_generation_prompt(coding_gen_template)

print("Registered Generation Prompts:")
for name in registry.list_generation_prompts():
    template = registry.get_generation_prompt(name)
    print(f"  - {name}: {template.num_candidates} candidates")

print("\nRegistered Evaluation Prompts:")
for name in registry.list_evaluation_prompts():
    template = registry.get_evaluation_prompt(name)
    print(f"  - {name}: {len(template.criteria)} criteria")

### 6.4 Building Actual Prompts with Variables

In [ ]:
# Example 1: Generate math candidates
print("\n" + "="*70)
print("EXAMPLE 1: Math Problem - Generation Prompt")
print("="*70)

math_gen = registry.get_generation_prompt("game_of_24")
problem = "[6, 6, 6, 6]"
context = {"Goal": "Make 24 from four 6s"}
prompt = math_gen.build_prompt(problem, context)
print(prompt[:500] + "...\n[truncated]")

# Example 2: Evaluate code debugging hypothesis
print("\n" + "="*70)
print("EXAMPLE 2: Code Debugging - Evaluation Prompt")
print("="*70)

coding_gen = registry.get_generation_prompt("code_debugging")
buggy_code = """
def find_max(numbers):
    max_val = 0
    for num in numbers:
        if num > max_val:
            max_val = num
    return max_val

result = find_max([-5, -3, -10])  # Returns 0, but should return -3
"""

context = {
    "Expected": "Returns -3 (the maximum of negative numbers)",
    "Actual": "Returns 0",
    "Issue Category": "Logic error in initialization"
}

gen_prompt = coding_gen.build_prompt(buggy_code, context)
print(gen_prompt)

## 7. Complete Working Example

Let's build a complete Tree of Thoughts example with actual node evaluation and tree expansion.

### 7.1 Data Structures for Tree Nodes

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List
from enum import Enum
import uuid
from datetime import datetime

class NodeStatus(Enum):
    """Status of a tree node."""
    PENDING = "pending"  # Generated but not evaluated
    EVALUATED = "evaluated"  # Has a score
    EXPANDED = "expanded"  # Has children
    PRUNED = "pruned"  # Marked for removal
    TERMINAL = "terminal"  # Is a solution

@dataclass
class ThoughtNode:
    """
    A single node in the tree of thoughts.
    
    Attributes:
        id: Unique identifier
        thought: The actual thought/solution step
        depth: How deep in the tree (0 = root)
        parent_id: ID of parent node (None for root)
        children_ids: IDs of child nodes
        score: Evaluation score (0-1 or None if not evaluated)
        status: Current status of the node
        metadata: Additional information (evaluation details, reasoning, etc)
        created_at: When this node was created
    """
    thought: str
    depth: int
    parent_id: Optional[str] = None
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    children_ids: List[str] = field(default_factory=list)
    score: Optional[float] = None
    status: NodeStatus = NodeStatus.PENDING
    metadata: Dict = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def add_child(self, child_id: str):
        """Add a child node ID."""
        if child_id not in self.children_ids:
            self.children_ids.append(child_id)
    
    def set_evaluation(self, score: float, details: Dict):
        """Set evaluation score and mark as evaluated."""
        if not (0 <= score <= 1):
            raise ValueError(f"Score must be 0-1, got {score}")
        self.score = score
        self.status = NodeStatus.EVALUATED
        self.metadata["evaluation"] = details
    
    def mark_pruned(self, reason: str):
        """Mark node as pruned."""
        self.status = NodeStatus.PRUNED
        self.metadata["pruned_reason"] = reason
    
    def is_promising(self, threshold: float = 0.5) -> bool:
        """Check if node meets threshold for expansion."""
        if self.score is None:
            return False
        return self.score >= threshold
    
    def __repr__(self) -> str:
        status_str = f" [{self.status.value}]"
        if self.score is not None:
            status_str += f" score={self.score:.2f}"
        return f"Node({self.id}{status_str}): {self.thought[:50]}..."

class ThoughtTree:
    """
    Manages a tree of thoughts.
    """
    
    def __init__(self, problem: str):
        self.problem = problem
        self.nodes = {}
        self.root_id = None
    
    def create_root(self, thought: str) -> ThoughtNode:
        """Create root node."""
        node = ThoughtNode(thought=thought, depth=0)
        self.root_id = node.id
        self.nodes[node.id] = node
        return node
    
    def add_child(self, parent_id: str, thought: str) -> ThoughtNode:
        """Add a child node."""
        if parent_id not in self.nodes:
            raise ValueError(f"Parent {parent_id} not found")
        
        parent = self.nodes[parent_id]
        child = ThoughtNode(
            thought=thought,
            depth=parent.depth + 1,
            parent_id=parent_id
        )
        
        self.nodes[child.id] = child
        parent.add_child(child.id)
        parent.status = NodeStatus.EXPANDED
        
        return child
    
    def get_node(self, node_id: str) -> ThoughtNode:
        """Get node by ID."""
        if node_id not in self.nodes:
            raise ValueError(f"Node {node_id} not found")
        return self.nodes[node_id]
    
    def get_nodes_by_status(self, status: NodeStatus) -> List[ThoughtNode]:
        """Get all nodes with given status."""
        return [n for n in self.nodes.values() if n.status == status]
    
    def get_promising_nodes(self, threshold: float = 0.5) -> List[ThoughtNode]:
        """Get all nodes that meet score threshold."""
        evaluated = self.get_nodes_by_status(NodeStatus.EVALUATED)
        return [n for n in evaluated if n.is_promising(threshold)]
    
    def get_best_path(self) -> List[ThoughtNode]:
        """Get path with highest scores."""
        if not self.root_id:
            return []
        
        best_path = []
        current_node = self.nodes[self.root_id]
        
        while current_node:
            best_path.append(current_node)
            
            if not current_node.children_ids:
                break
            
            # Find child with highest score
            best_child_id = max(
                current_node.children_ids,
                key=lambda cid: self.nodes[cid].score or 0
            )
            current_node = self.nodes[best_child_id]
        
        return best_path
    
    def get_depth_statistics(self) -> Dict:
        """Get stats about tree depth."""
        depths = [n.depth for n in self.nodes.values()]
        return {
            "num_nodes": len(self.nodes),
            "max_depth": max(depths) if depths else 0,
            "min_depth": min(depths) if depths else 0,
        }
    
    def __repr__(self) -> str:
        stats = self.get_depth_statistics()
        return f"ThoughtTree({stats['num_nodes']} nodes, max_depth={stats['max_depth']})"

print("✓ ThoughtNode and ThoughtTree classes defined")
print("  - ThoughtNode: Individual thought in the tree")
print("  - ThoughtTree: Manages the complete tree structure")

### 7.2 Mock LLM Evaluator (Simulating API Calls)

In [ ]:
class MockLLMEvaluator:
    """
    Simulates LLM evaluation without making actual API calls.
    In production, this would call OpenAI, Claude, etc.
    """
    
    def __init__(self, seed: int = 42):
        """Initialize with optional seed for reproducibility."""
        import random
        self.random = random.Random(seed)
    
    def evaluate_math_solution(self, problem: str, candidate: str) -> tuple:
        """
        Evaluate a math solution candidate.
        Returns (score, evaluation_details)
        """
        # Simple heuristic: if it mentions the target number, higher score
        score = 0.3  # Base score
        details = {"criterion": "math_solution_viability"}
        
        # Simulate evaluation logic
        if "24" in candidate or "24.0" in candidate:
            score += 0.5
            details["mentions_target"] = True
        else:
            details["mentions_target"] = False
        
        if "Step" in candidate or "step" in candidate:
            score += 0.1
            details["has_steps"] = True
        else:
            details["has_steps"] = False
        
        # Add some randomness for realistic simulation
        score = min(1.0, score + self.random.uniform(-0.05, 0.05))
        
        return score, details
    
    def evaluate_event_planning(self, problem: str, candidate: str) -> tuple:
        """
        Evaluate an event planning candidate.
        """
        score = 0.4
        details = {"criterion": "event_planning_feasibility"}
        
        # Check for specific planning elements
        if "budget" in candidate.lower():
            score += 0.2
            details["has_budget"] = True
        
        if "activity" in candidate.lower() or "activities" in candidate.lower():
            score += 0.15
            details["has_activities"] = True
        
        if "venue" in candidate.lower():
            score += 0.15
            details["has_venue"] = True
        
        score = min(1.0, score + self.random.uniform(-0.05, 0.05))
        return score, details
    
    def generate_candidates(
        self, 
        problem: str, 
        num_candidates: int = 3,
        domain: str = "general"
    ) -> List[str]:
        """
        Simulate generating candidate solutions.
        In production, this would be an LLM API call.
        """
        
        if domain == "math":
            candidates = [
                "Try dividing one number by a fractional result: 8 / (3 - 8/3)",
                "Multiply pairs and combine: (3 + 3) * (8 - 4)",
                "Use factorials and operations: (3!) * (8 - 4)"
            ]
        elif domain == "event_planning":
            candidates = [
                "Venue: Bowling alley. Activity: Bowling tournament with pizza dinner. Budget: Lanes $1000, Food $800, Prizes $200.",
                "Venue: Escape room facility. Activity: Teams rotate rooms, then group dinner. Budget: Rooms $1200, Dinner $800.",
                "Venue: Conference center. Activity: Workshop morning, team challenges afternoon. Budget: Space $300, Catering $1400, Materials $300."
            ]
        else:
            candidates = [
                f"Approach 1: Start with breaking down the problem",
                f"Approach 2: Look for patterns or similarities",
                f"Approach 3: Consider constraints and work backwards"
            ]
        
        return candidates[:num_candidates]

# Initialize the mock evaluator
evaluator = MockLLMEvaluator(seed=42)

print("✓ MockLLMEvaluator defined")
print("  - Simulates LLM evaluation without API calls")
print("  - Would be replaced with real API calls in production")

### 7.3 Tree Building Algorithm

In [ ]:
class TreeOfThoughtsBuilder:
    """
    Orchestrates the tree building process.
    """
    
    def __init__(
        self,
        evaluator: MockLLMEvaluator,
        max_depth: int = 3,
        candidates_per_node: int = 3,
        score_threshold: float = 0.5,
        verbose: bool = True
    ):
        self.evaluator = evaluator
        self.max_depth = max_depth
        self.candidates_per_node = candidates_per_node
        self.score_threshold = score_threshold
        self.verbose = verbose
    
    def build_tree(self, problem: str, domain: str = "general") -> ThoughtTree:
        """
        Build a complete tree of thoughts for a problem.
        """
        tree = ThoughtTree(problem)
        
        if self.verbose:
            print(f"\n{'='*70}")
            print(f"Building Tree of Thoughts for: {problem[:60]}...")
            print(f"{'='*70}")
        
        # Create root node with the problem
        root = tree.create_root(f"Problem: {problem}")
        if self.verbose:
            print(f"\n[LEVEL 0] Root: {root.id}")
            print(f"  Problem: {problem}")
        
        # Queue for BFS expansion
        queue = [root]
        level = 0
        
        while queue and level < self.max_depth:
            level += 1
            next_queue = []
            
            if self.verbose:
                print(f"\n[LEVEL {level}] Processing {len(queue)} nodes...")
            
            for parent_node in queue:
                if self.verbose:
                    print(f"\n  Parent: {parent_node.id}")
                
                # Generate candidates
                candidates = self.evaluator.generate_candidates(
                    parent_node.thought,
                    num_candidates=self.candidates_per_node,
                    domain=domain
                )
                
                # Evaluate each candidate
                for i, candidate_thought in enumerate(candidates):
                    # Create child node
                    child = tree.add_child(parent_node.id, candidate_thought)
                    
                    # Evaluate child
                    if domain == "math":
                        score, details = self.evaluator.evaluate_math_solution(
                            problem, candidate_thought
                        )
                    elif domain == "event_planning":
                        score, details = self.evaluator.evaluate_event_planning(
                            problem, candidate_thought
                        )
                    else:
                        # Default: random score
                        import random
                        score = random.random()
                        details = {}
                    
                    child.set_evaluation(score, details)
                    
                    if self.verbose:
                        viability = "✓ PROMISING" if child.is_promising(self.score_threshold) else "✗ LOW SCORE"
                        print(f"    Child {i+1} ({child.id}): score={score:.2f} {viability}")
                        print(f"      Thought: {candidate_thought[:60]}...")
                    
                    # Add to next queue if promising
                    if child.is_promising(self.score_threshold) and level < self.max_depth:
                        next_queue.append(child)
            
            queue = next_queue
        
        if self.verbose:
            stats = tree.get_depth_statistics()
            print(f"\n{'='*70}")
            print(f"Tree Complete:")
            print(f"  Total nodes: {stats['num_nodes']}")
            print(f"  Max depth: {stats['max_depth']}")
            print(f"{'='*70}")
        
        return tree

print("✓ TreeOfThoughtsBuilder defined")
print("  - Manages tree construction")
print("  - Controls depth, branching, and thresholds")

### 7.4 Running the Complete Example

In [ ]:
# Example 1: Build a tree for a math problem
print("\n\n")
print("#" * 70)
print("# EXAMPLE 1: Math Problem - Make 24 from [3, 8, 3, 8]")
print("#" * 70)

builder = TreeOfThoughtsBuilder(
    evaluator=evaluator,
    max_depth=2,
    candidates_per_node=3,
    score_threshold=0.5,
    verbose=True
)

math_tree = builder.build_tree(
    problem="Make 24 from [3, 8, 3, 8]",
    domain="math"
)

print(f"\n\nMath Tree Structure: {math_tree}")

### 7.5 Analyzing the Built Tree

In [ ]:
def print_tree_summary(tree: ThoughtTree):
    """
    Print a detailed summary of the built tree.
    """
    print("\n" + "="*70)
    print("TREE SUMMARY")
    print("="*70)
    
    # Stats
    stats = tree.get_depth_statistics()
    print(f"\nBasic Stats:")
    print(f"  Total Nodes: {stats['num_nodes']}")
    print(f"  Max Depth: {stats['max_depth']}")
    
    # Nodes by status
    print(f"\nNodes by Status:")
    for status in NodeStatus:
        nodes = tree.get_nodes_by_status(status)
        print(f"  {status.value.capitalize()}: {len(nodes)}")
    
    # Score distribution
    evaluated_nodes = tree.get_nodes_by_status(NodeStatus.EVALUATED)
    if evaluated_nodes:
        scores = [n.score for n in evaluated_nodes if n.score is not None]
        if scores:
            print(f"\nScore Statistics:")
            print(f"  Min Score: {min(scores):.2f}")
            print(f"  Max Score: {max(scores):.2f}")
            print(f"  Avg Score: {sum(scores)/len(scores):.2f}")
    
    # Promising nodes
    promising = tree.get_promising_nodes(threshold=0.5)
    print(f"\nPromising Nodes (score >= 0.5): {len(promising)}")
    for node in promising:
        print(f"  {node}")
    
    # Best path
    print(f"\nBest Path Found:")
    best_path = tree.get_best_path()
    for i, node in enumerate(best_path):
        indent = "  " * node.depth
        score_str = f" [score={node.score:.2f}]" if node.score else ""
        print(f"{indent}Level {node.depth}: {node.id}{score_str}")
        if node.depth > 0:  # Don't print full root
            print(f"{indent}  └─ {node.thought[:70]}...")

# Analyze the math tree
print_tree_summary(math_tree)

### 7.6 Second Example: Event Planning

In [ ]:
# Example 2: Build a tree for event planning
print("\n\n")
print("#" * 70)
print("# EXAMPLE 2: Event Planning - Company Outing")
print("#" * 70)

event_builder = TreeOfThoughtsBuilder(
    evaluator=evaluator,
    max_depth=2,
    candidates_per_node=3,
    score_threshold=0.5,
    verbose=True
)

event_tree = event_builder.build_tree(
    problem="Plan company outing: 50 people, $2000 budget, indoor, team-building",
    domain="event_planning"
)

# Analyze the event planning tree
print_tree_summary(event_tree)

### 7.7 Visualizing the Difference: In-Context vs Agentic ToT

In [ ]:
print("\n\n")
print("="*70)
print("COMPARING: In-Context ToT vs Agentic ToT")
print("="*70)

print("""
With the code we just built, we've demonstrated AGENTIC ToT:

Our Implementation:
  ✓ Separate function calls for generation (evaluator.generate_candidates)
  ✓ Separate function calls for evaluation (evaluator.evaluate_*)
  ✓ Agent controls the exploration (TreeOfThoughtsBuilder)
  ✓ Explicit tree data structures (ThoughtTree, ThoughtNode)
  ✓ Full visibility into all decisions
  ✓ Can control depth, branching, and thresholds
  ✓ Can restart from any node
  ✓ Can implement search algorithms (BFS, DFS, beam search)

If We Used In-Context ToT Instead:
  ✓ Single prompt to LLM ("Use ToT to solve...")
  ✓ LLM internally simulates the tree
  ✓ Returns one final answer
  ✗ Can't see intermediate nodes
  ✗ Can't control exploration strategy
  ✗ Can't restart from specific branches
  ✗ Limited by token context window

For Production Systems:
  → If problem is complex: Use Agentic (what we built)
  → If problem is simple: Use In-Context (cheaper, faster)
  → If you need reproducibility: Use Agentic
  → If you need cost efficiency: Use In-Context
""")

print("\nOur Agentic Implementation Benefits:")
print(f"  • Complete visibility: We saw every node and score")
print(f"  • Reproducible: Same seed produces same tree")
print(f"  • Analyzable: We can extract best paths, statistics, etc.")
print(f"  • Controllable: We set depth, candidates, thresholds")
print(f"  • Integrable: Can add external data, constraints, etc.")

### 7.8 Key Learnings

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║                     KEY LEARNINGS FROM PHASE 1                      ║
╚════════════════════════════════════════════════════════════════════╝

1. TREE OF THOUGHTS FUNDAMENTALS
   ├─ ToT explores multiple solution paths (unlike CoT which is linear)
   ├─ Trees allow backtracking and alternative exploration
   ├─ Evaluation of candidates must precede deeper exploration
   └─ Trade-off between depth and quality depends on problem

2. AGENT vs MODEL APPROACHES
   ├─ In-Context ToT: Single LLM call, model simulates tree internally
   │   └─ Use for: Simple problems, cost-sensitive, quick iteration
   ├─ Agentic ToT: Program orchestrates multiple LLM calls
   │   └─ Use for: Complex problems, reproducibility, control
   └─ Comparison: Speed vs Control, Cost vs Quality

3. PROMPT ENGINEERING MATTERS
   ├─ Generation prompts: Be specific about structure and diversity
   ├─ Evaluation prompts: Define clear criteria without requiring solutions
   ├─ Domain-specific: Different prompts for math vs writing vs coding
   └─ Templates: Reusable prompt structures improve consistency

4. PRACTICAL IMPLEMENTATION
   ├─ Use data structures to represent tree (ThoughtNode, ThoughtTree)
   ├─ Separate concerns: Generation, Evaluation, Expansion
   ├─ Track node status: PENDING → EVALUATED → EXPANDED/PRUNED
   └─ Use thresholds to guide exploration (not all nodes expand)

5. CONTROLLING THE SEARCH
   ├─ Max depth: Prevent infinite exploration
   ├─ Score threshold: Only expand promising branches
   ├─ Candidates per node: Control branching factor
   ├─ Search algorithm: BFS, DFS, beam search for different strategies
   └─ Pruning: Remove low-scoring branches early

6. WHEN ToT SUCCEEDS
   ✓ Problems with multiple valid paths
   ✓ Problems where intermediate evaluation is easier than full solving
   ✓ Complex problems requiring planning and lookahead
   ✓ Domains where creativity/diversity matters
   ✓ When quality is more important than speed/cost

7. WHEN ToT MIGHT NOT HELP
   ✗ Problems with single obvious solution path
   ✗ Problems where evaluation requires complete solution
   ✗ Extremely simple problems (CoT is sufficient)
   ✗ Real-time applications (too slow)
   ✗ Budget-constrained applications (many LLM calls)

╔════════════════════════════════════════════════════════════════════╗
║                     NEXT STEPS (Phase 2+)                          ║
╚════════════════════════════════════════════════════════════════════╝

Phase 2 will cover:
  • Implementing different search strategies (BFS, DFS, beam search)
  • Pruning strategies for efficient exploration
  • Real API integration (OpenAI, Anthropic)
  • Advanced prompt optimization
  • Cost analysis and optimization

Phase 3 will cover:
  • Multi-level reasoning (nested trees)
  • Hybrid approaches (CoT + ToT)
  • Integration with knowledge bases and tools
  • Production considerations
  • Performance benchmarking
""")

## Summary

This enhanced Phase 1 notebook has covered:

1. **Introduction to Tree of Thoughts** - Why trees are better than chains for complex problems

2. **Agent vs Model Approaches** - Two ways to implement ToT with detailed trade-offs
   - In-Context ToT: Simple, single API call, less control
   - Agentic ToT: Complex, multiple calls, full control

3. **Prompt Engineering for Generation** - How to create good prompts that generate diverse, relevant candidates
   - Bad prompts (generic, vague)
   - Good prompts (specific, structured)
   - Domain-specific variations

4. **Prompt Engineering for Evaluation** - How to evaluate candidates without full solutions
   - Evaluation without complete solving
   - Multiple evaluation criteria
   - Scoring rubrics and guidelines
   - Scoring examples across domains

5. **Side-by-Side Comparisons** - CoT vs ToT on the same problems
   - How CoT takes one linear path
   - How ToT explores multiple branches
   - Visual comparison of decision trees

6. **Practical Code Examples** - Reusable classes and templates
   - GenerationPromptTemplate
   - EvaluationPromptTemplate
   - PromptRegistry for managing multiple prompts

7. **Complete Working Example** - End-to-end implementation
   - ThoughtNode and ThoughtTree data structures
   - MockLLMEvaluator for testing
   - TreeOfThoughtsBuilder for orchestration
   - Two complete examples (math and event planning)

You now have a solid foundation for understanding and implementing Tree of Thoughts in both in-context and agentic forms!